In [74]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa
from scipy.stats import norm
from sklearn.metrics import cohen_kappa_score


# Preparation

In [75]:
new1 = pd.read_csv('new1.csv')
new2 = pd.read_csv('new2.csv')
old1 = pd.read_csv('old1.csv')
old2 = pd.read_csv('old2.csv')

In [76]:
new1.drop(columns=['комментарий'], inplace=True, errors='ignore')
new1['tag'] = new1['tag'].str.lower()
new1

,example,quote,change,tag
0,В последним предложении он пишет про нас можно...,состовляем,составляем,ortho
1,Я сразу выхожу из комнати и откриваю холодильн...,комнати,комнаты,ortho
2,NaN,откриваю,открываю,ortho
3,NaN,ведеть,видеть,ortho
4,И она дала отпор смерте.,смерте,смерти,ortho
5,В начале двадцатого века в Америке продажа ако...,алкогола,алкоголя,infl
6,"В 1963 он был аресторан, и в следущом году был...",аресторан,арестован,ortho
7,NaN,следущом,следующем,ortho
8,"Парни боятся, что их выганют из школы, но всё ...",выганют,выгонят,ortho
9,Первие признаки осеньи видны на деревьях золот...,осеньи,осени,infl


In [77]:
new2.drop(columns=['Unnamed: 4'], inplace=True, errors='ignore')
new2['tag'] = new2['tag'].str.lower()
new2

,example,quote,change,tag
0,В последним предложении он пишет про нас можно...,последним,последнем,ortho
1,В последним предложении он пишет про нас можно...,состовляем,составляем,ortho
2,Я сразу выхожу из комнати и откриваю холодильн...,комнати,комнаты,ortho
3,Я сразу выхожу из комнати и откриваю холодильн...,откриваю,открываю,ortho
4,И она дала отпор смерте.,смерте,смерти,infl
5,В начале двадцатого века в Америке продажа ако...,акогола,алкоголя,ortho
6,"В 1963 он был аресторан, и в следущом году был...",аресторан,арестован,ortho
7,"В 1963 он был аресторан, и в следущом году был...",следущом,следующем,infl
8,"Парни боятся, что их выганют из школы, но всё ...",выганют,выгонят,ortho
9,Первие признаки осеньи видны на деревьях золот...,первие,первые,ortho


In [78]:

old1.drop(columns=['комментарий', 'доп. тег ortho'], inplace=True, errors='ignore')
old1["tag"] = old1["tag"].str.lower()
old1

,example,quote,change,tag
0,В последним предложении он пишет про нас можно...,последним,последнем,ortho
1,NaN,состовляем,составляем,ortho
2,Я сразу выхожу из комнати и откриваю холодильн...,комнати,комнаты,infl
3,NaN,откриваю,открываю,ortho
4,NaN,ведеть,видеть,ortho
...,...,...,...,...
56,Все люди ходят в хорошом настроений.,хорошом,хорошем,infl
57,NaN,настроений,настроении,gov
58,Мой любимой вид спорта- настольный теннис.,любимой,любимый,gov
59,"Многое, что мы знаем, передалось нам от древне...",древнех,древних,infl


In [79]:
old2.drop(columns=['комментарии', 'доп. тег'], inplace=True)
old2["tag"] = old2["tag"].str.lower()
old2

,example,quote,change,tag
0,В последним предложении он пишет про нас можно...,состовляем,составляем,ortho
1,В последним предложении он пишет про нас можно...,последним,последнем,infl
2,Я сразу выхожу из комнати и откриваю холодильн...,комнати,комнаты,gov
3,Я сразу выхожу из комнати и откриваю холодильн...,откриваю,открываю,ortho
4,Я сразу выхожу из комнати и откриваю холодильн...,ведеть,видеть,ortho
...,...,...,...,...
63,Все люди ходят в хорошом настроений.,хорошом,хорошем,infl
64,Все люди ходят в хорошом настроений.,настроений,настроении,gov
65,Мой любимой вид спорта- настольный теннис.,любимой,любимый,ortho
66,"Многое, что мы знаем, передалось нам от древне...",древнех,древних,ortho


# Analysis

## Old

In [80]:
merged_old = pd.merge(old1[['quote', 'tag']], old2[['quote', 'tag']], on="quote", suffixes=("_1", "_2"))
merged_old

,quote,tag_1,tag_2
0,последним,ortho,infl
1,состовляем,ortho,ortho
2,комнати,infl,gov
3,откриваю,ortho,ortho
4,ведеть,ortho,ortho
5,смерте,ortho,infl
6,акогола,ortho,ortho
7,акогола,infl,ortho
8,аресторан,ortho,ortho
9,следущом,ortho,ortho


In [81]:
all_labels = sorted(set(merged_old["tag_1"]) | set(merged_old["tag_2"]))

table_old = []
for _, row in merged_old.iterrows():
    counts = [0] * len(all_labels)
    counts[all_labels.index(row["tag_1"])] += 1
    counts[all_labels.index(row["tag_2"])] += 1
    table_old.append(counts)

table_old = pd.DataFrame(table_old, columns=all_labels)

kappa_old = fleiss_kappa(table_old.values)
print(f"Fleiss' kappa: {kappa_old:.4f}")


Fleiss' kappa: 0.4955


In [82]:
print(f"Cohen's kappa: {cohen_kappa_score(merged_old["tag_1"], merged_old["tag_2"]):.4f}")

Cohen's kappa: 0.4965


## New

In [84]:
merged_new = pd.merge(new1[['quote', 'tag']], new2[['quote', 'tag']], on="quote", suffixes=("_1", "_2"))
merged_new


,quote,tag_1,tag_2
0,состовляем,ortho,ortho
1,комнати,ortho,ortho
2,откриваю,ortho,ortho
3,смерте,ortho,infl
4,аресторан,ortho,ortho
5,следущом,ortho,infl
6,выганют,ortho,ortho
7,лишного,infl,infl
8,рабочых,ortho,ortho
9,обучениям,ortho,gov


In [85]:
all_labels = sorted(set(merged_new["tag_1"]) | set(merged_new["tag_2"]))

table_new = []
for _, row in merged_new.iterrows():
    counts = [0] * len(all_labels)
    counts[all_labels.index(row["tag_1"])] += 1
    counts[all_labels.index(row["tag_2"])] += 1
    table_new.append(counts)

table_new = pd.DataFrame(table_new, columns=all_labels)

kappa_new = fleiss_kappa(table_new.values)
print(f"Fleiss' kappa: {kappa_new:.4f}")

Fleiss' kappa: 0.5411


In [86]:
print(f"Cohen's kappa: {cohen_kappa_score(merged_new["tag_1"], merged_new["tag_2"]):.4f}")

Cohen's kappa: 0.5455


## Statistical tests

For Fleiss' kappa

In [88]:
def fleiss_se(table):
    N, k = table.shape
    n = table.sum(axis=1)[0]
    P_bar = table.sum(axis=0) / (N * n)
    P_e = np.sum(P_bar**2)
    kappa = fleiss_kappa(table)
    # оценка дисперсии по Fleiss (1979)
    var = (2 * (N - 1) * (P_e * (1 - P_e))) / (N * n * (n - 1) * (1 - P_e)**2)
    return np.sqrt(var)


se_old = fleiss_se(table_old)
se_new = fleiss_se(table_new)

z = (kappa_new - kappa_old) / np.sqrt(se_old**2 + se_new**2)
p_value = norm.sf(abs(z))

print(f"κ old: {kappa_old:.4f}, SE old: {se_old:.4f}")
print(f"κ new: {kappa_new:.4f}, SE new: {se_new:.4f}")
print(f"z: {z:.4f}, p-value: {p_value:.4f}")

κ old: 0.4955, SE old: 0.8305
κ new: 0.5411, SE new: 0.9273
z: 0.0367, p-value: 0.4854


For Cohen's kappa

In [89]:
def cohen_se(tags1, tags2):
    N = len(tags1)
    po = np.mean(tags1 == tags2)
    labels1 = np.unique(tags1)
    labels2 = np.unique(tags2)
    pe = sum([(np.mean(tags1 == l1) * np.mean(tags2 == l1)) for l1 in labels1])
    kappa = (po - pe) / (1 - pe)
    se = np.sqrt(po * (1 - po) / (N * (1 - pe)**2))
    return kappa, se

kappa_old, se_old = cohen_se(merged_old['tag_1'].values, merged_old['tag_2'].values)
kappa_new, se_new = cohen_se(merged_new['tag_1'].values, merged_new['tag_2'].values)

z = (kappa_new - kappa_old) / np.sqrt(se_old**2 + se_new**2)
p_value = norm.sf(abs(z))

print(f"κ old: {kappa_old:.4f}, SE old: {se_old:.4f}")
print(f"κ new: {kappa_new:.4f}, SE new: {se_new:.4f}")
print(f"z: {z:.4f}, p-value: {p_value:.4f}")

κ old: 0.4965, SE old: 0.1056
κ new: 0.5455, SE new: 0.1250
z: 0.2992, p-value: 0.3824
